# Structured streaming Demo

Nicolas Rihana et Benjamin Labos - groupe 4

Nous reprenons la démo du cours pour suivre les modifications de Wikipédia
en français avec Kafka et Spark. Nous ajoutons une analyse de la taille des
modifications par minute et une analyse de l'activité par page avec des fenêtres
qui se chevauchent.

Source : [consignes et fichiers du cours](https://github.com/adaltas/ece-big-data-processing-fall-2026/tree/main/05.structured-streaming).

**Exécution** : lancer `admin.py`, puis le producteur avant « Run All ».
Les tableaux ci-dessous et les CSV sont calculés à partir des événements Kafka.
Le notebook fonctionne pendant 45 secondes par défaut, puis arrête ses requêtes.
Les événements restent disponibles dans Kafka pour une nouvelle exécution.

In [1]:
# Imports cell
import csv
import json
import os
from pathlib import Path
import time
import uuid

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    abs as spark_abs, avg, col, count, from_json, from_unixtime,
    lit, max as spark_max, sum as spark_sum, when, window,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, BooleanType,
)
import pandas as pd
from IPython.display import display


In [2]:
# initialize the spark session
# Match the Kafka connector to the installed Spark and Scala versions.
spark_home = Path(os.environ.get("SPARK_HOME", "/usr/local/spark"))
release = (spark_home / "RELEASE").read_text().split()
spark_version = release[1]
scala_jar = next((spark_home / "jars").glob("scala-library-*.jar"))
scala_binary = ".".join(scala_jar.stem.removeprefix("scala-library-").split(".")[:2])
kafka_package = f"org.apache.spark:spark-sql-kafka-0-10_{scala_binary}:{spark_version}"

spark = (SparkSession.builder
    .master("local[2]")
    .appName("KafkaStreamingPrototype")
    .config("spark.jars.packages", kafka_package)
    .config("spark.jars.repositories", "https://repo.maven.apache.org/maven2")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Stop only this notebook's previous queries before rerunning it.
for previous in spark.streams.active:
    if (previous.name or "").startswith("wiki_"):
        previous.stop()

data_dir = Path("/home/jovyan/data")
results_dir = data_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
run_id = uuid.uuid4().hex
checkpoint_dir = data_dir / "checkpoints" / run_id
finite_run = os.getenv("WIKISTREAM_FINITE", "0") == "1"
run_seconds = int(os.getenv("WIKISTREAM_RUN_SECONDS", "45"))
print(f"Spark {spark.version}; connector {kafka_package}; UTC; finite_run={finite_run}")


Spark 4.2.0; connector org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0; UTC; finite_run=True


In [3]:
# Define the initial raw streaming dataframe
raw_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:29092"))
    .option("subscribe", os.getenv("WIKISTREAM_TOPIC", "wikistreams"))
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", "10000")
    .load())
print("Streaming source:", raw_df.isStreaming)


Streaming source: True


In [4]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", LongType(), True),
        StructField("new", LongType(), True),
    ]), True),
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

# Event time comes from Wikimedia, not the processing clock.
events_df = (parsed_df
    .filter((col("type") == "edit") & col("timestamp").isNotNull() & col("title").isNotNull())
    .withColumn("event_time", from_unixtime(col("timestamp")).cast("timestamp"))
    .withColumn("bytes_delta", col("length.new") - col("length.old"))
    .withColumn("bytes_added", when(col("bytes_delta") > 0, col("bytes_delta")).otherwise(lit(0)))
    .withColumn("bytes_removed", when(col("bytes_delta") < 0, -col("bytes_delta")).otherwise(lit(0)))
    .withWatermark("event_time", "2 minutes"))
events_df.printSchema()


root
 |-- title: string (nullable = true)
 |-- user: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- type: string (nullable = true)
 |-- wiki: string (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- comment: string (nullable = true)
 |-- length: struct (nullable = true)
 |    |-- old: long (nullable = true)
 |    |-- new: long (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- bytes_delta: long (nullable = true)
 |-- bytes_added: long (nullable = true)
 |-- bytes_removed: long (nullable = true)



In [5]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = (events_df
    .groupBy(window(col("event_time"), "1 hour"), col("bot"))
    .agg(count("*").alias("edit_count"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"), col("edit_count")))


In [6]:
# Use console output (goes to Docker logs)
def configured_writer(df, name):
    writer = (df.writeStream
        .queryName(name)
        .outputMode("update")
        .option("checkpointLocation", str(checkpoint_dir / name)))
    return writer.trigger(availableNow=True) if finite_run else writer.trigger(processingTime="10 seconds")

streaming_query_df = (configured_writer(windowed_counts_df, "wiki_demo_bots")
    .format("console")
    .option("truncate", False)
    .start())
print("Demo started:", streaming_query_df.name)


Demo started: wiki_demo_bots


## Exploration 1 — fenêtres fixes (tumbling), taille des modifications

Nous choisissons des fenêtres de **1 minute**, sans chevauchement : `[début, fin)`.
Ce format permet de comparer deux minutes successives sans compter plusieurs fois
la même modification. Nous séparons les modifications signalées comme venant
d'un robot (`bot=True`) des autres (`bot=False`).

Nous calculons le nombre de modifications, la variation nette, les octets ajoutés
et supprimés, ainsi que la taille moyenne absolue et la plus grande modification.

`length.new - length.old` mesure la variation de longueur de la page en octets,
pas le volume réseau ni la taille totale du diff. Si une longueur manque,
la variation reste inconnue (`null`) et les moyennes l'ignorent ;
`edits_with_size` permet de repérer cette différence de couverture.

In [7]:
tumbling_df = (events_df
    .groupBy(window("event_time", "1 minute"), "wiki", "bot")
    .agg(
        count("*").alias("edit_count"),
        count("bytes_delta").alias("edits_with_size"),
        spark_sum("bytes_delta").alias("net_bytes"),
        spark_sum("bytes_added").alias("added_bytes"),
        spark_sum("bytes_removed").alias("removed_bytes"),
        avg(spark_abs(col("bytes_delta"))).alias("avg_abs_bytes"),
        spark_max(spark_abs(col("bytes_delta"))).alias("max_abs_bytes"),
    )
    .select(col("window.start").alias("window_start"),
            col("window.end").alias("window_end"), "wiki", "bot", "edit_count",
            "edits_with_size", "net_bytes", "added_bytes", "removed_bytes",
            "avg_abs_bytes", "max_abs_bytes"))


## Exploration 2 — fenêtres glissantes (overlapping), activité par page

Nous choisissons une fenêtre de **2 minutes** qui avance de **30 secondes**.
Deux fenêtres consécutives se chevauchent donc pendant 90 secondes, et une
modification appartient à **4 fenêtres**. Ce choix permet de suivre les pages
actives sur une période plus longue, tout en actualisant les résultats souvent.
Nous calculons le nombre de modifications et la variation d'octets par page.
Les comptes des fenêtres glissantes ne doivent pas être additionnés comme
s'il s'agissait d'événements distincts.

In [8]:
sliding_df = (events_df
    .groupBy(window("event_time", "2 minutes", "30 seconds"), "wiki", "title")
    .agg(count("*").alias("edit_count"),
         spark_sum("bytes_delta").alias("net_bytes"),
         spark_sum(spark_abs(col("bytes_delta"))).alias("absolute_bytes"))
    .select(col("window.start").alias("window_start"),
            col("window.end").alias("window_end"), "wiki", "title",
            "edit_count", "net_bytes", "absolute_bytes"))


## Exécution des explorations et export des résultats

Le mode `update` fournit les groupes modifiés à chaque micro-lot.
Le callback les remplace par leur nouvelle valeur dans les CSV : il ne les
additionne pas une seconde fois. Chaque fichier contient les résultats de
l'exécution courante et s'actualise à chaque micro-lot non vide.
Les tableaux sont également écrits dans les logs du conteneur.

Le watermark de **2 minutes** autorise le nettoyage de l'état des fenêtres
terminées. Il suit le plus grand temps d'événement observé, avec ce retard ;
il ne représente pas un délai d'attente du processeur. Des événements trop
anciens peuvent être ignorés. Les CSV conservent les groupes déjà observés.
Le sink CSV est un export pédagogique ; il ne garantit pas une livraison
exactly-once de bout en bout après panne. Chaque nouvelle exécution utilise
un nouveau checkpoint et relit le topic depuis le début.

In [9]:
snapshots = {}

def csv_and_console_sink(name, group_columns, fieldnames):
    snapshots[name] = {}
    # Initialize this run's export even when no message matches the filters.
    with (results_dir / f"{name}.csv").open("w", newline="", encoding="utf-8") as handle:
        csv.DictWriter(handle, fieldnames=fieldnames).writeheader()
    def save_batch(batch_df, batch_id):
        batch_df.persist()
        try:
            rows = batch_df.collect()
            if not rows:
                return
            for row in rows:
                record = row.asDict()
                key = tuple(record[field] for field in group_columns)
                snapshots[name][key] = record
            destination = results_dir / f"{name}.csv"
            temporary = destination.with_suffix(".tmp")
            with temporary.open("w", newline="", encoding="utf-8") as handle:
                writer = csv.DictWriter(handle, fieldnames=batch_df.columns)
                writer.writeheader()
                writer.writerows(snapshots[name].values())
            temporary.replace(destination)
            print(f"{name}: batch={batch_id}, updated_groups={len(rows)}", flush=True)
            batch_df.orderBy(col("edit_count").desc()).show(10, truncate=50)
            # Mirror a short sample to the container stdout for docker compose logs.
            try:
                with Path("/proc/1/fd/1").open("a", encoding="utf-8") as docker_log:
                    print(f"{name}: batch={batch_id}, updated_groups={len(rows)}", file=docker_log)
                    for row in rows[:10]:
                        print(json.dumps(row.asDict(), ensure_ascii=False, default=str), file=docker_log)
            except OSError as exc:
                print(f"Docker log output unavailable: {exc}")
        finally:
            batch_df.unpersist()
    return save_batch

tumbling_query = (configured_writer(tumbling_df, "wiki_tumbling")
    .foreachBatch(csv_and_console_sink("tumbling_1min", ["window_start", "window_end", "wiki", "bot"], tumbling_df.columns))
    .start())
sliding_query = (configured_writer(sliding_df, "wiki_sliding")
    .foreachBatch(csv_and_console_sink("sliding_2min_30s", ["window_start", "window_end", "wiki", "title"], sliding_df.columns))
    .start())
queries = [streaming_query_df, tumbling_query, sliding_query]
print("Queries started:", [query.name for query in queries])


Queries started: ['wiki_demo_bots', 'wiki_tumbling', 'wiki_sliding']


In [10]:
# Finite validation drains the current Kafka offsets; interactive runs stay bounded.
try:
    if finite_run:
        for query in queries:
            if not query.awaitTermination(240):
                raise TimeoutError(f"Query did not finish: {query.name}")
    else:
        deadline = time.monotonic() + run_seconds
        while time.monotonic() < deadline:
            for query in queries:
                if query.exception() is not None:
                    raise RuntimeError(str(query.exception()))
            time.sleep(1)
finally:
    for query in queries:
        if query.isActive:
            query.stop()

progress = {}
for query in queries:
    batches = [json.loads(p.json) if hasattr(p, "json") else p for p in query.recentProgress]
    progress[query.name] = {
        "input_rows": sum(p["numInputRows"] for p in batches),
        "batches": len(batches),
        "exception": str(query.exception()) if query.exception() else None,
    }
(results_dir / "validation.json").write_text(
    json.dumps({"spark_version": spark.version, "finite_run": finite_run,
                "queries": progress}, indent=2), encoding="utf-8")
print(json.dumps(progress, indent=2))
print("All notebook queries stopped:", not any(q.isActive for q in queries))


sliding_2min_30s: batch=0, updated_groups=106


tumbling_1min: batch=0, updated_groups=4


+-------------------+-------------------+------+------------------------------------------+----------+---------+--------------+
|       window_start|         window_end|  wiki|                                     title|edit_count|net_bytes|absolute_bytes|
+-------------------+-------------------+------+------------------------------------------+----------+---------+--------------+
|2026-10-16 09:00:00|2026-10-16 09:02:00|frwiki|Catégorie:Saison des Lakers de Los Angeles|         3|       85|           111|
|2026-10-16 08:59:00|2026-10-16 09:01:00|frwiki|Catégorie:Saison des Lakers de Los Angeles|         3|       85|           111|
|2026-10-16 08:59:30|2026-10-16 09:01:30|frwiki|Catégorie:Saison des Lakers de Los Angeles|         3|       85|           111|
|2026-10-16 08:59:30|2026-10-16 09:01:30|frwiki|            Utilisateur:Futsuli/Brouillon2|         2|     -194|           358|
|2026-10-16 08:59:00|2026-10-16 09:01:00|frwiki|            Utilisateur:Futsuli/Brouillon2|         2|  

+-------------------+-------------------+------+-----+----------+---------------+---------+-----------+-------------+------------------+-------------+
|       window_start|         window_end|  wiki|  bot|edit_count|edits_with_size|net_bytes|added_bytes|removed_bytes|     avg_abs_bytes|max_abs_bytes|
+-------------------+-------------------+------+-----+----------+---------------+---------+-----------+-------------+------------------+-------------+
|2026-10-16 08:59:00|2026-10-16 09:00:00|frwiki|false|        14|             14|     2682|       2766|           84|203.57142857142858|         2219|
|2026-10-16 09:00:00|2026-10-16 09:01:00|frwiki|false|        10|             10|     -801|        116|          917|             103.3|          618|
|2026-10-16 09:00:00|2026-10-16 09:01:00|frwiki| true|         3|              3|      525|        842|          317| 386.3333333333333|          713|
|2026-10-16 08:59:00|2026-10-16 09:00:00|frwiki| true|         2|              2|      870|   

{
  "wiki_demo_bots": {
    "input_rows": 29,
    "batches": 2,
    "exception": null
  },
  "wiki_tumbling": {
    "input_rows": 29,
    "batches": 2,
    "exception": null
  },
  "wiki_sliding": {
    "input_rows": 29,
    "batches": 2,
    "exception": null
  }
}
All notebook queries stopped: True


In [11]:
for name in ["tumbling_1min", "sliding_2min_30s"]:
    records = list(snapshots[name].values())
    print(f"\n{name}: {len(records)} groups")
    if records:
        table = pd.DataFrame(records).sort_values(["edit_count", "window_start"], ascending=[False, True])
        display(table.head(15))
    else:
        print("No event processed in this run. Check the producer and run the notebook again.")

if snapshots["tumbling_1min"]:
    fixed = pd.DataFrame(snapshots["tumbling_1min"].values())
    sliding = pd.DataFrame(snapshots["sliding_2min_30s"].values())
    total = int(fixed.edit_count.sum())
    bot_edits = int(fixed.loc[fixed.bot == True, "edit_count"].sum())
    print(f"\nBilan : {total} modifications, dont {bot_edits} par des robots ({100 * bot_edits / total:.1f} %).")
    print(f"Variation nette : {int(fixed.net_bytes.sum())} octets ; "
          f"ajoutés : {int(fixed.added_bytes.sum())} ; supprimés : {int(fixed.removed_bytes.sum())}.")
    print(f"Comptes des fenêtres glissantes : {int(sliding.edit_count.sum())} "
          f"pour {total} événements retenus dans les fenêtres fixes.")



tumbling_1min: 4 groups


,window_start,window_end,wiki,bot,edit_count,edits_with_size,net_bytes,added_bytes,removed_bytes,avg_abs_bytes,max_abs_bytes
2,2026-10-16 08:59:00,2026-10-16 09:00:00,frwiki,False,14,14,2682,2766,84,203.571429,2219
3,2026-10-16 09:00:00,2026-10-16 09:01:00,frwiki,False,10,10,-801,116,917,103.300000,618
1,2026-10-16 09:00:00,2026-10-16 09:01:00,frwiki,True,3,3,525,842,317,386.333333,713
0,2026-10-16 08:59:00,2026-10-16 09:00:00,frwiki,True,2,2,870,870,0,435.000000,825



sliding_2min_30s: 106 groups


,window_start,window_end,wiki,title,edit_count,net_bytes,absolute_bytes
94,2026-10-16 08:59:00,2026-10-16 09:01:00,frwiki,Catégorie:Saison des Lakers de Los Angeles,3,85,111
38,2026-10-16 08:59:30,2026-10-16 09:01:30,frwiki,Catégorie:Saison des Lakers de Los Angeles,3,85,111
93,2026-10-16 09:00:00,2026-10-16 09:02:00,frwiki,Catégorie:Saison des Lakers de Los Angeles,3,85,111
39,2026-10-16 08:58:30,2026-10-16 09:00:30,frwiki,Catégorie:Saison des Lakers de Los Angeles,2,98,98
68,2026-10-16 08:58:30,2026-10-16 09:00:30,frwiki,Utilisateur:Futsuli/Brouillon2,2,-194,358
67,2026-10-16 08:59:00,2026-10-16 09:01:00,frwiki,Utilisateur:Futsuli/Brouillon2,2,-194,358
66,2026-10-16 08:59:30,2026-10-16 09:01:30,frwiki,Utilisateur:Futsuli/Brouillon2,2,-194,358
3,2026-10-16 08:57:30,2026-10-16 08:59:30,frwiki,Pete Seeger,1,309,309
6,2026-10-16 08:57:30,2026-10-16 08:59:30,frwiki,Savary de Mauléon,1,4,4
8,2026-10-16 08:57:30,2026-10-16 08:59:30,frwiki,2009 en natation,1,-21,21



Bilan : 29 modifications, dont 5 par des robots (17.2 %).
Variation nette : 3276 octets ; ajoutés : 4594 ; supprimés : 1318.
Comptes des fenêtres glissantes : 116 pour 29 événements retenus dans les fenêtres fixes.


## Ce que nous observons

Sur l'échantillon enregistré le **16 octobre 2026**, nous obtenons **29 modifications distinctes**
de Wikipédia en français : **5** sont signalées comme venant de robots et **24**
ne le sont pas. Le champ `bot=False` ne permet pas de prouver que la modification
est entièrement manuelle.

Les fenêtres fixes montrent **16 modifications entre 08:59 et 09:00 UTC**, puis
**13 entre 09:00 et 09:01 UTC**. Parmi les modifications non signalées comme bots,
la variation nette passe de **+2 682** à **−801 octets**. Une minute active peut
donc aussi correspondre à une diminution de la taille des pages. Au total,
**4 594 octets** sont ajoutés et **1 318 supprimés**, soit une variation nette
de **+3 276 octets**.

Dans les fenêtres glissantes, la page **« Catégorie:Saison des Lakers de Los Angeles »**
atteint **3 modifications** dans une fenêtre de deux minutes. Elle apparaît dans
plusieurs fenêtres voisines parce que celles-ci se chevauchent : cela ne représente
pas des modifications différentes à chaque ligne. Nous retrouvons **116 appartenances**
pour **29 événements**, ce qui correspond bien aux quatre fenêtres par événement.

Les événements sont identifiés par `meta.id`. Nous vérifions l'absence de doublons
dans cet échantillon pour ne pas confondre un message renvoyé avec une nouvelle
modification. Le producteur mémorise les identifiants déjà présents dans son fichier
local et ceux reçus pendant la capture. Il faut lancer un seul producteur par topic.

Cet échantillon est court et ne suffit pas pour conclure sur l'activité habituelle
de Wikipédia ou sur la part générale des robots. Les minutes aux extrémités ne
sont pas forcément observées entièrement. Une nouvelle capture peut donner des
résultats différents des observations ci-dessus.

## Filtres du producteur

Le filtrage est fait dans `wikistream_producer.py`, **avant Kafka**. Par défaut,
le producteur conserve les événements `type=edit` de `fr.wikipedia.org`.
La fonction `matches` applique les critères de langue, de titre et de robot.

```bash
# Follow French and English Wikipedia for two minutes.
python wikistream_producer.py --languages fr,en --duration 120

# Follow one exact title in French Wikipedia, excluding bots.
python wikistream_producer.py --languages fr --page "Paris" --exclude-bots --duration 120
```

Avec `--languages all`, on conserve les modifications de tous les projets Wikimedia.
Les filtres sont cumulables ; une page peu modifiée peut ne produire aucun événement.
Changer de filtre ne supprime pas les messages déjà présents dans Kafka. Pour
comparer deux captures séparément, il faut utiliser des topics différents.

Les requêtes s'arrêtent en fin de notebook. Pour poursuivre l'observation,
relancer le notebook pendant que le producteur fonctionne ou augmenter
`WIKISTREAM_RUN_SECONDS` avant l'exécution.
